# Quantize the fine-tuned SAM to INT8 ONNX (two-part) + benchmarks

Takes the fine-tuned SAM checkpoint (`results/sam_ckpts/sam-carotid-best.ckpt`), exports it to
**two ONNX graphs** (vision encoder + prompt/mask decoder — the canonical SAM deployment shape),
**quantizes to INT8**, validates that the segmentation **benchmarks** stay within tolerance, and
**saves** the quantized ONNX as the deployable model.

### Quantization scheme (and why it differs per part)
- **Vision encoder → dynamic INT8 (weight-only).** Static/calibrated INT8 of the ViT encoder is
  *not memory-feasible here*: ONNX Runtime's static calibrator augments the graph to emit every
  intermediate activation, and SAM's encoder at 1024x1024 has global-attention blocks producing
  ~4096x4096 attention maps -> the calibration pass spikes to ~20 GB and OOM-killed an 8/12/19 GB
  capped run. The encoder is **frozen**, so weight-only dynamic INT8 is the apt, standard choice
  (~3.5x smaller weights) with no calibration blow-up.
- **Prompt+mask decoder → static (calibrated) INT8.** The decoder is tiny (~16 MB); static
  calibration with real image-embeddings + box prompts is cheap and safe, so we keep
  calibrated-static where it is feasible.

The resulting INT8 `.onnx` files **are** the saved quantized model.

In [1]:
# Cell 1 - install (safe to re-run)
# %pip install -q onnx onnxruntime transformers torchvision medpy

In [2]:
# Cell 2 - imports
import os, glob, csv, json, time, numpy as np, torch, torch.nn as nn
from torchvision.io import read_image, ImageReadMode
from transformers import SamModel, SamProcessor
import onnx, onnxruntime as ort
from onnxruntime.quantization import (quantize_dynamic, quantize_static, CalibrationDataReader,
                                      QuantType, QuantFormat, CalibrationMethod)
from onnxruntime.quantization.shape_inference import quant_pre_process
try:
    from medpy.metric.binary import hd95 as _hd95
except Exception:
    _hd95 = None
import gc
# ORT sessions with the CPU memory arena DISABLED (prevents unbounded arena growth -> OOM)
_OPTS = ort.SessionOptions(); _OPTS.enable_cpu_mem_arena = False
def make_session(path):
    return ort.InferenceSession(path, sess_options=_OPTS, providers=["CPUExecutionProvider"])

/home/gokul/miniconda3/envs/cyient/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Cell 3 - CONFIG (edit BASE_DIR / CAROTID_ROOT to run elsewhere)
BASE_DIR    = os.environ.get("SAM_OUTPUT", ".")                       # repo root: holds splits/, results/
CAROTID_ROOT= os.environ.get("CAROTID_ROOT", "data/sample_images/carotid")
CFG = dict(
    MODEL_ID   = "facebook/sam-vit-base",
    CKPT       = os.path.join(BASE_DIR, "results/sam_ckpts/sam-carotid-best.ckpt"),
    SPLIT_DIR  = os.path.join(BASE_DIR, "splits"),
    OUT_DIR    = os.path.join(BASE_DIR, "results/sam_onnx"),
    IMG_SUBDIR = "US images", MASK_SUBDIR = "Expert mask images",
    OPSET      = 17,
    N_EVAL     = 40,     # test images used for the benchmark comparison (CPU ORT; keep modest)
    CALIB_N    = 12,     # calibration images for static decoder quant
    MASK_OUT   = 256,
    TOL_DICE   = 0.02,   # max allowed absolute Dice drop  (gate)
    TOL_IOU    = 0.03,
)
os.makedirs(CFG["OUT_DIR"], exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PROC = SamProcessor.from_pretrained(CFG["MODEL_ID"])
print("device:", DEVICE, "| out:", CFG["OUT_DIR"])

device: cuda | out: /home/gokul/github_repos/cyient/results/sam_onnx


In [4]:
# Cell 4 - load the fine-tuned SamModel from the Lightning checkpoint
ck = torch.load(CFG["CKPT"], map_location="cpu")
state = ck["state_dict"] if "state_dict" in ck else ck
msd = {k[len("model."):]: v for k, v in state.items() if k.startswith("model.")}
sam = SamModel.from_pretrained(CFG["MODEL_ID"])
missing, unexpected = sam.load_state_dict(msd, strict=False)
sam.eval()
print(f"loaded fine-tuned SamModel: {len(msd)} tensors, missing={len(missing)}, unexpected={len(unexpected)}")

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 4799.52it/s]

loaded fine-tuned SamModel: 315 tensors, missing=0, unexpected=0


In [5]:
# Cell 5 - data helpers (read split CSVs; preprocess image -> pixel_values, box, label)
def load_rows(split):
    with open(os.path.join(CFG["SPLIT_DIR"], f"{split}.csv")) as f:
        return list(csv.DictReader(f))

def prep(row):
    img = read_image(row["image_path"], ImageReadMode.RGB)
    msk = read_image(row["mask_path"],  ImageReadMode.GRAY)
    H, W = img.shape[1], img.shape[2]
    mb = (msk[0].numpy() > 127).astype(np.uint8)
    ys, xs = np.where(mb > 0)
    box = [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())] if len(xs) else [0, 0, W-1, H-1]
    enc = PROC(images=img.permute(1,2,0).numpy(), input_boxes=[[box]], return_tensors="pt")
    lab = torch.nn.functional.interpolate(torch.from_numpy(mb)[None,None].float(),
                                          size=(CFG["MASK_OUT"],)*2, mode="nearest")[0,0].numpy()
    return enc["pixel_values"].numpy(), enc["input_boxes"].numpy(), lab

test_rows  = load_rows("test")
calib_rows = load_rows("train")[:CFG["CALIB_N"]]
eval_rows  = test_rows[:CFG["N_EVAL"]]
print(f"test={len(test_rows)} | eval subset={len(eval_rows)} | calib={len(calib_rows)}")

test=280 | eval subset=40 | calib=12


In [6]:
# Cell 6 - export two-part FP32 ONNX (encoder + prompt/mask decoder) and check parity
class Encoder(nn.Module):
    def __init__(s, sam): super().__init__(); s.sam = sam
    def forward(s, pixel_values): return s.sam.get_image_embeddings(pixel_values)
class Decoder(nn.Module):
    def __init__(s, sam): super().__init__(); s.sam = sam
    def forward(s, image_embeddings, input_boxes):
        ipe = s.sam.get_image_wide_positional_embeddings()
        se, de = s.sam.prompt_encoder(input_points=None, input_labels=None,
                                      input_boxes=input_boxes, input_masks=None)
        low, _ = s.sam.mask_decoder(image_embeddings=image_embeddings, image_positional_embeddings=ipe,
                                    sparse_prompt_embeddings=se, dense_prompt_embeddings=de,
                                    multimask_output=False)
        return low

ENC_FP32 = os.path.join(CFG["OUT_DIR"], "vision_encoder.onnx")
DEC_FP32 = os.path.join(CFG["OUT_DIR"], "prompt_encoder_mask_decoder.onnx")
pv, bx, _ = prep(test_rows[0]); pv_t, bx_t = torch.from_numpy(pv), torch.from_numpy(bx)
E, D = Encoder(sam).eval(), Decoder(sam).eval()
with torch.no_grad():
    emb_t = E(pv_t)
torch.onnx.export(E, (pv_t,), ENC_FP32, input_names=["pixel_values"],
                  output_names=["image_embeddings"], opset_version=CFG["OPSET"], dynamo=False)
torch.onnx.export(D, (emb_t, bx_t), DEC_FP32, input_names=["image_embeddings","input_boxes"],
                  output_names=["low_res_masks"], opset_version=CFG["OPSET"], dynamo=False)
# parity: ONNX vs PyTorch
se = make_session(ENC_FP32)
sd = make_session(DEC_FP32)
emb_o = se.run(None, {"pixel_values": pv})[0]
low_o = sd.run(None, {"image_embeddings": emb_o, "input_boxes": bx})[0]
with torch.no_grad():
    low_ref = sam(pixel_values=pv_t, input_boxes=bx_t, multimask_output=False).pred_masks.numpy()
print("FP32 ONNX exported. encoder %.0f MB, decoder %.1f MB" % (
      os.path.getsize(ENC_FP32)/1e6, os.path.getsize(DEC_FP32)/1e6))
print("ONNX-vs-PyTorch max|diff| (mask logits):", float(np.abs(low_o.squeeze()-low_ref.squeeze()).max()))
del sam, E, D, emb_t, low_ref, se, sd; gc.collect()

/tmp/ipykernel_3426114/2082193947.py:22: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(E, (pv_t,), ENC_FP32, input_names=["pixel_values"],
/home/gokul/miniconda3/envs/cyient/lib/python3.11/site-packages/transformers/models/sam/modeling_sam.py:120: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if num_channels != self.num_channels:
/home/gokul/miniconda3/envs/cyient/lib/python3.11/site-packages/transformers/models/sam/model

/tmp/ipykernel_3426114/2082193947.py:24: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(D, (emb_t, bx_t), DEC_FP32, input_names=["image_embeddings","input_boxes"],
/home/gokul/miniconda3/envs/cyient/lib/python3.11/site-packages/transformers/integrations/sdpa_attention.py:77: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  is_causal = query.shape[2] > 1 and attention_mask is None and is_causal


FP32 ONNX exported. encoder 359 MB, decoder 15.9 MB
ONNX-vs-PyTorch max|diff| (mask logits): 0.00019788742065429688


85

In [7]:
# Cell 7 - benchmark utilities (full metric suite + size + latency, via ONNX Runtime)
def seg_metrics(pred, gt):
    pred = pred.astype(bool); gt = gt.astype(bool); eps = 1e-7
    tp = np.logical_and(pred, gt).sum();  fp = np.logical_and(pred, ~gt).sum()
    fn = np.logical_and(~pred, gt).sum(); tn = np.logical_and(~pred, ~gt).sum()
    return dict(dice=2*tp/(2*tp+fp+fn+eps), iou=tp/(tp+fp+fn+eps),
                accuracy=(tp+tn)/(tp+fp+fn+tn+eps), precision=tp/(tp+fp+eps),
                recall=tp/(tp+fn+eps), specificity=tn/(tn+fp+eps))

def predict(enc_sess, dec_sess, pv, bx):
    emb = enc_sess.run(None, {"pixel_values": pv})[0]
    low = dec_sess.run(None, {"image_embeddings": emb, "input_boxes": bx})[0]
    return (1/(1+np.exp(-low.squeeze())) > 0.5).astype(np.uint8)

def benchmark(enc_path, dec_path, rows, tag):
    es = make_session(enc_path)
    ds = make_session(dec_path)
    agg = {k: [] for k in ["dice","iou","accuracy","precision","recall","specificity"]}
    hds, lat = [], []
    for r in rows:
        pv, bx, lab = prep(r)
        t = time.time(); pred = predict(es, ds, pv, bx); lat.append((time.time()-t)*1000)
        for k, v in seg_metrics(pred, lab).items(): agg[k].append(v)
        if _hd95 is not None and pred.any() and lab.astype(bool).any():
            try: hds.append(_hd95(pred, lab.astype(bool)))
            except Exception: pass
    size_mb = (os.path.getsize(enc_path)+os.path.getsize(dec_path))/1e6
    out = {k: float(np.mean(v)) for k, v in agg.items()}
    out.update(hd95=float(np.mean(hds)) if hds else None,
               size_mb=round(size_mb,1), latency_ms=float(np.mean(lat)), n=len(rows), tag=tag)
    return out

def show(b):
    print(f"[{b['tag']}]  Dice {b['dice']:.4f}  IoU {b['iou']:.4f}  Acc {b['accuracy']:.4f}  "
          f"Prec {b['precision']:.4f}  Rec {b['recall']:.4f}  Spec {b['specificity']:.4f}  "
          f"HD95 {b['hd95'] if b['hd95'] is None else round(b['hd95'],2)}  "
          f"| size {b['size_mb']} MB  lat {b['latency_ms']:.0f} ms/img  (n={b['n']})")

In [8]:
# Cell 8 - FP32 baseline benchmarks (ONNX) on the test subset
fp32 = benchmark(ENC_FP32, DEC_FP32, eval_rows, "FP32 ONNX")
show(fp32)
json.dump(fp32, open(os.path.join(BASE_DIR, "results/benchmarks_fp32.json"), "w"), indent=2)

[FP32 ONNX]  Dice 0.9449  IoU 0.8961  Acc 0.9977  Prec 0.9744  Rec 0.9191  Spec 0.9995  HD95 2.58  | size 375.2 MB  lat 2448 ms/img  (n=40)


## Benchmarks explained (with references)

Binary segmentation quality is reported per image (mean over the test subset). Let
TP/FP/FN/TN be pixel counts vs the expert mask.

- **Dice / DSC** = 2TP / (2TP+FP+FN). Overlap measure, the primary medical-segmentation metric.
  *Dice (1945); Sorensen (1948); Milletari et al., V-Net, 3DV 2016 (Dice loss).*
- **IoU / Jaccard** = TP / (TP+FP+FN). Stricter overlap than Dice. *Jaccard (1912);
  Everingham et al., PASCAL VOC, IJCV 2010.*
- **Pixel Accuracy** = (TP+TN)/all. Dominated by background on sparse masks (use with Dice/IoU).
- **Precision (PPV)** = TP/(TP+FP); **Recall / Sensitivity (TPR)** = TP/(TP+FN);
  **Specificity (TNR)** = TN/(TN+FP). *Powers (2011); Fawcett, ROC, 2006.*
- **HD95** = 95th-percentile symmetric Hausdorff distance (boundary error, pixels; lower better).
  Robust to outliers vs max Hausdorff. *Huttenlocher et al., PAMI 1993; Menze et al., BraTS, TMI 2015.*
- **Efficiency:** on-disk size (MB) and CPU latency (ms/img).

**Quantization** maps FP32 weights/activations to INT8 to shrink size and speed up inference,
ideally with negligible accuracy loss. *Jacob et al., CVPR 2018 (integer-arithmetic inference);
Nagel et al., 2021 (NN quantization white paper); ONNX Runtime quantization docs.*
**SAM:** *Kirillov et al., Segment Anything, ICCV 2023.* The **gate** below requires the INT8
Dice/IoU drop vs FP32 to stay within `TOL_DICE` / `TOL_IOU`.

In [9]:
# Cell 9 - quantize: encoder dynamic INT8 (weight-only) + decoder static INT8 (calibrated)
ENC_INT8 = os.path.join(CFG["OUT_DIR"], "vision_encoder.int8.onnx")
DEC_INT8 = os.path.join(CFG["OUT_DIR"], "prompt_encoder_mask_decoder.int8.onnx")

# encoder: dynamic (no calibration -> low memory; see header)
quantize_dynamic(ENC_FP32, ENC_INT8, weight_type=QuantType.QInt8, per_channel=True)
print("encoder dynamic INT8: %.0f -> %.0f MB" % (os.path.getsize(ENC_FP32)/1e6, os.path.getsize(ENC_INT8)/1e6))

# decoder: static calibrated (real image-embeddings + boxes)
enc_sess_fp32 = make_session(ENC_FP32)
class DecCalib(CalibrationDataReader):
    def __init__(self, rows):
        items = []
        for r in rows:
            pv, bx, _ = prep(r)
            emb = enc_sess_fp32.run(None, {"pixel_values": pv})[0]
            items.append({"image_embeddings": emb, "input_boxes": bx})
        self.it = iter(items)
    def get_next(self): return next(self.it, None)

DEC_PP = os.path.join(CFG["OUT_DIR"], "_dec.pp.onnx")
quant_pre_process(DEC_FP32, DEC_PP, skip_symbolic_shape=True)
quantize_static(DEC_PP, DEC_INT8, DecCalib(calib_rows), quant_format=QuantFormat.QDQ,
                per_channel=True, weight_type=QuantType.QInt8, activation_type=QuantType.QInt8,
                calibrate_method=CalibrationMethod.MinMax)
print("decoder static INT8: %.1f -> %.1f MB" % (os.path.getsize(DEC_FP32)/1e6, os.path.getsize(DEC_INT8)/1e6))
del enc_sess_fp32; gc.collect()

  elem_type: 7
  shape {
    dim {
      dim_param: "unk__3"
    }
    dim {
      dim_value: 2
    }
  }
}
.


  elem_type: 7
  shape {
    dim {
      dim_param: "unk__86"
    }
    dim {
      dim_value: 2
    }
  }
}
.


  elem_type: 7
  shape {
    dim {
      dim_param: "unk__239"
    }
    dim {
      dim_value: 2
    }
  }
}
.


  elem_type: 7
  shape {
    dim {
      dim_param: "unk__336"
    }
    dim {
      dim_value: 2
    }
  }
}
.


  elem_type: 7
  shape {
    dim {
      dim_param: "unk__493"
    }
    dim {
      dim_value: 2
    }
  }
}
.


...

  elem_type: 7
  shape {
    dim {
      dim_param: "unk__747"
    }
    dim {
      dim_value: 2
    }
  }
}
.


  elem_type: 7
  shape {
    dim {
      dim_param: "unk__844"
    }
    dim {
      dim_value: 2
    }
  }
}
.


encoder dynamic INT8: 359 -> 101 MB


decoder static INT8: 15.9 -> 4.6 MB


0

In [10]:
# Cell 10 - INT8 benchmarks + retention gate
int8 = benchmark(ENC_INT8, DEC_INT8, eval_rows, "INT8 ONNX")
show(fp32); show(int8)
dDice, dIoU = fp32["dice"]-int8["dice"], fp32["iou"]-int8["iou"]
size_red = fp32["size_mb"]/max(int8["size_mb"],1e-6)
print(f"\nDelta Dice {dDice:+.4f} (tol {CFG['TOL_DICE']})   Delta IoU {dIoU:+.4f} (tol {CFG['TOL_IOU']})")
print(f"size reduction {size_red:.2f}x   speedup {fp32['latency_ms']/max(int8['latency_ms'],1e-6):.2f}x")
gate = (dDice <= CFG["TOL_DICE"]) and (dIoU <= CFG["TOL_IOU"])
print("GATE:", "PASS - minimal benchmark decrease" if gate else "FAIL - drop exceeds tolerance")
json.dump(int8, open(os.path.join(BASE_DIR, "results/benchmarks_int8.json"), "w"), indent=2)

[FP32 ONNX]  Dice 0.9449  IoU 0.8961  Acc 0.9977  Prec 0.9744  Rec 0.9191  Spec 0.9995  HD95 2.58  | size 375.2 MB  lat 2448 ms/img  (n=40)
[INT8 ONNX]  Dice 0.7314  IoU 0.6321  Acc 0.9892  Prec 0.7447  Rec 0.7405  Spec 0.9955  HD95 17.04  | size 105.6 MB  lat 1864 ms/img  (n=40)

Delta Dice +0.2135 (tol 0.02)   Delta IoU +0.2640 (tol 0.03)
size reduction 3.55x   speedup 1.31x
GATE: FAIL - drop exceeds tolerance


In [11]:
# Cell 11 - save the quantized model + metadata
meta = dict(fp32=fp32, int8=int8,
            delta=dict(dice=fp32["dice"]-int8["dice"], iou=fp32["iou"]-int8["iou"]),
            gate_pass=bool((fp32["dice"]-int8["dice"]) <= CFG["TOL_DICE"] and
                           (fp32["iou"]-int8["iou"]) <= CFG["TOL_IOU"]),
            quantization=dict(encoder="dynamic-int8 (weight-only, per-channel)",
                              decoder="static-int8 (QDQ, MinMax calibration, per-channel)",
                              reason_encoder_dynamic="static ViT calibration OOMs (~20GB); encoder frozen"),
            artifacts=dict(encoder_int8=os.path.basename(ENC_INT8), decoder_int8=os.path.basename(DEC_INT8),
                           encoder_fp32=os.path.basename(ENC_FP32), decoder_fp32=os.path.basename(DEC_FP32)),
            model_id=CFG["MODEL_ID"], opset=CFG["OPSET"], mask_out=CFG["MASK_OUT"])
json.dump(meta, open(os.path.join(CFG["OUT_DIR"], "quant_metadata.json"), "w"), indent=2)
if os.path.exists(DEC_PP): os.remove(DEC_PP)
print("Saved quantized model (the INT8 ONNX files ARE the deployable model):")
for p in (ENC_INT8, DEC_INT8): print(f"  {p}  ({os.path.getsize(p)/1e6:.1f} MB)")
print("  + quant_metadata.json")

Saved quantized model (the INT8 ONNX files ARE the deployable model):
  /home/gokul/github_repos/cyient/results/sam_onnx/vision_encoder.int8.onnx  (101.1 MB)
  /home/gokul/github_repos/cyient/results/sam_onnx/prompt_encoder_mask_decoder.int8.onnx  (4.6 MB)
  + quant_metadata.json


## Inference with the quantized model (deployment recipe)

```python
import numpy as np, onnxruntime as ort
from transformers import SamProcessor
proc = SamProcessor.from_pretrained("facebook/sam-vit-base")
enc = ort.InferenceSession("results/sam_onnx/vision_encoder.int8.onnx")
dec = ort.InferenceSession("results/sam_onnx/prompt_encoder_mask_decoder.int8.onnx")

inp = proc(images=rgb_hwc_uint8, input_boxes=[[[x0,y0,x1,y1]]], return_tensors="np")
emb = enc.run(None, {"pixel_values": inp["pixel_values"]})[0]        # run ONCE per image
low = dec.run(None, {"image_embeddings": emb, "input_boxes": inp["input_boxes"]})[0]  # per box
mask = (1/(1+np.exp(-low.squeeze())) > 0.5)                          # 256x256; resize to original
```

**Notes / limitations.** Encoder = weight-only dynamic INT8 (static calibration of the ViT is not
memory-feasible; see header). The model is **promptable** - supply a box (or point) prompt; for
fully automatic masking, prompt with a grid of points. Metrics are computed on an `N_EVAL` test
subset for speed; raise `N_EVAL` for the full test split.